In [114]:
import pandas as pd 
df = pd.read_csv('tech_layoffs_til_2025.csv')

In [115]:
df.head()

,Nr,Company,Location_HQ,Region,USState,Country,Continent,Laid_Off,Date_layoffs,Percentage,Company_Size_before_Layoffs,Company_Size_after_layoffs,Industry,Stage,Money_Raised_in__mil,Year,latitude,longitude
0,1,Tamara Mellon,Los Angeles,other,California,USA,North America,20.0,2020-03-12,40.0,50.0,30.0,Retail,Series C,90.0,2020,34.053691,-118.242766
1,2,HopSkipDrive,Los Angeles,other,California,USA,North America,8.0,2020-03-13,10.0,80.0,72.0,Transportation,Unknown,45.0,2020,34.053691,-118.242766
2,3,Panda Squad,San Francisco,San Francisco Bay Area,California,USA,North America,6.0,2020-03-13,75.0,8.0,2.0,Consumer,Seed,1.0,2020,37.779259,-122.419329
3,4,Help.com,Austin,other,Texas,USA,North America,16.0,2020-03-16,100.0,16.0,0.0,Support,Seed,6.0,2020,30.271129,-97.743700
4,5,Inspirato,Denver,other,Colorado,USA,North America,130.0,2020-03-16,22.0,591.0,461.0,Travel,Series C,79.0,2020,39.739236,-104.984862


In [116]:
df.isnull().mean() * 100

Nr                              0.000000
Company                         0.000000
Location_HQ                     0.000000
Region                          0.000000
USState                         0.041459
Country                         0.000000
Continent                       0.000000
Laid_Off                       15.422886
Date_layoffs                    0.000000
Percentage                     18.615257
Company_Size_before_Layoffs    26.658375
Company_Size_after_layoffs     23.009950
Industry                        0.000000
Stage                           6.799337
Money_Raised_in__mil           15.091211
Year                            0.000000
latitude                        0.000000
longitude                       0.000000
dtype: float64

In [117]:
df['Industry'].value_counts()

Industry
Finance                         289
Retail                          176
Healthcare                      156
Transportation                  142
Consumer                        134
                               ... 
Advertising Services              1
Meat Technologies                 1
E-Learning                        1
Conversational chatbot tools      1
Insurance Software Solutions      1
Name: count, Length: 118, dtype: int64

### Note -- Cleaning To-Do
1. `Industry` column has 118 categories; many appear only 1-2 times. 
   Will group low-frequency categories into "Other" during cleaning step.

In [118]:
df['Company'].value_counts()

Company
Google              16
Amazon              16
Microsoft           13
Intel               10
Meta                 8
                    ..
Bitdefender          1
Crystal Dynamics     1
Fanatics             1
Couchbase            1
PitchBook            1
Name: count, Length: 1713, dtype: int64

### Analysis Strategy Note
- Company column has long-tail distribution (1713 unique companies, 
  most appear only once). 
- Company-level analysis (e.g., "top companies by layoffs") will focus 
  on companies with sufficient occurrences (e.g., top 15-20 by frequency) 
  to avoid drawing conclusions from single, potentially anomalous events.
- Industry-level analysis will use the full dataset, as aggregation 
  across companies reduces noise from individual events.

In [119]:
df['Year'].value_counts()

Year
2024    694
2022    568
2023    474
2025    332
2020    330
2021     14
Name: count, dtype: int64

### 2021 Anomaly Explained
- `Year` 2021 shows only 14 layoff events, dramatically lower than all 
  other years (330–694). This is NOT a data quality issue.
- Real-world context: 2021 was a hiring boom fueled by near-zero interest 
  rates post-pandemic. Companies (Meta, Amazon, Google) expanded headcount 
  aggressively, layoffs were nearly nonexistent.
- This over-hiring set up the 2022–2023 layoff wave: the Fed raised rates 
  sharply in 2022, funding got expensive, and companies corrected their 
  overstaffing.
- 2024–2025 layoffs differ in nature: less about correcting hiring mistakes, 
  more about AI-driven structural restructuring.
- Implication for analysis: the 2020–2025 trend is not one continuous 
  story — it's at least 3 distinct phases (pandemic shock →

In [120]:
df['Stage'].value_counts()

Stage
Post-IPO          574
Unknown           356
Series B          266
Series C          211
Series D          203
Acquired          176
Series E          123
Series A          116
Series F           66
Seed               49
Private Equity     37
Series H           27
Series G           19
Subsidiary         11
Series I            8
Series J            6
Name: count, dtype: int64

### Stage Column Insights

1. **"Unknown" should be treated as missing data**: Although "Unknown" 
   (356 records) is not a blank cell, it carries no usable information 
   about a company's funding stage. Treating it as a distinct category 
   in analysis would introduce bias, so it should be handled the same 
   way as missing values during cleaning.

2. **Post-IPO vs. early-stage layoffs carry very different risk signals**: 
   Post-IPO companies (574 records) remain financially stable even after 
   layoffs — these cuts typically reflect cost optimization, not distress. 
   In contrast, large-scale layoffs at early-stage companies (Seed, Series A/B) 
   may signal that the company is approaching failure or that its investors 
   have run out of capital — a much higher risk signal for job seekers.

In [121]:
df['Country'].value_counts()

Country
USA                        1557
India                       177
Israel                      122
Canada                      110
Germany                      81
UK                           74
Brazil                       48
Australia                    34
Singapore                    28
Sweden                       24
France                       13
Indonesia                    12
Nigeria                      11
Ireland                      10
Japan                         9
Netherlands                   8
Kenya                         8
China                         8
Norway                        5
United Kingdom                5
Denmark                       5
United Arabian Emirates       5
Estonia                       5
Spain                         4
Argentina                     4
Poland                        3
New Zealand                   3
Austria                       3
Finland                       3
Hong Kong                     3
Chile                         3


###  Country Distribution & Data Bias

USA accounts for 64.6% (1,557/2,412) of all layoff records. This is likely 
driven by two compounding factors:

1. **Real industry concentration**: The tech industry is genuinely concentrated 
   in the US (Silicon Valley, Seattle, etc.), so a higher share of layoffs 
   originating from the US is partly a true reflection of where the industry is based.

2. **Data collection bias**: This dataset relies primarily on English-language 
   media reporting and crowdsourced submissions (layoffs.fyi). Non-English-speaking 
   countries are structurally disadvantaged in this data collection method — 
   their layoffs are less likely to be captured even if they occur at similar rates.

**Implication for analysis**: Country-level comparisons should be interpreted 
as "US vs. other major reported tech hubs" rather than a complete global picture. 
This limitation will be explicitly noted in the BRD's Scope & Limitations section.

In [122]:
df['Company_Size_before_Layoffs'].value_counts()

Company_Size_before_Layoffs
1000.0    59
500.0     50
200.0     43
100.0     33
300.0     31
          ..
2635.0     1
815.0      1
347.0      1
4100.0     1
2447.0     1
Name: count, Length: 715, dtype: int64

In [123]:
df['Company_Size_before_Layoffs'].quantile([0.33, 0.67])

0.33     300.0
0.67    1002.8
Name: Company_Size_before_Layoffs, dtype: float64

### Company Size Binning Strategy

Initially proposed 3 size categories (≤300 / 301–1000 / >1000) based on 
business intuition. Since I lack extensive industry experience to confirm 
this judgment, I validated the cutoffs using quantile analysis 
(33rd and 67th percentiles of `Company_Size_before_Layoffs`).

Result: quantiles returned 300 and ~1003, closely matching the intuition-based 
cutoffs. Final categories:
- **Small**: ≤300 employees
- **Mid-size**: 301–1000 employees  
- **Large**: >1000 employees

This data-validated approach balances business interpretability (round, 
memorable cutoffs) with statistical grounding (cutoffs reflect actual 
data distribution, not arbitrary guesses).

In [124]:
df['Money_Raised_in__mil'].describe()

count      2048.000000
mean        683.777832
std        4292.579693
min           0.000000
25%          50.000000
50%         161.000000
75%         451.250000
max      121900.000000
Name: Money_Raised_in__mil, dtype: float64

The Money_Raised_in__mil column has a mean of 683.78 but a median of only 
161, which is a big gap. This means a small number of companies (probably 
big players like Google) have unusually high funding/valuation numbers that 
are pulling the average way up. For later analysis, I should use the median 
instead of the mean, otherwise the results will be skewed by these outliers.